# 01 · Data Understanding

**Project:** Data Analyst – Mental Health (Canada)
**Pipeline step:** 1 of 10 – Dataset selection / understanding
**Owner:** Samir · **Last run:** see output below

## Purpose
Load every raw dataset the team collected from `data/raw/`, profile it (shape, columns,
types, missing values, time coverage, geography), and write a machine-readable
**data inventory** to `data/processed/`. This notebook does **no cleaning** – it only
looks. Cleaning happens in `02_data_cleaning.ipynb`.

## What this notebook produces (the "test" outputs in `data/processed/`)
| File | Contents |
|---|---|
| `01_data_inventory.csv` | one row per dataset: rows, cols, time range, geography, % missing, notes |
| `01_column_profiles.csv` | one row per column across all datasets: dtype, % missing, # unique, sample values |
| `01_pipeline_check.txt` | timestamped marker proving `data/raw` → `data/processed` works end to end |


## 1 · Setup

In [1]:
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

# --- locate project root whether the notebook is run from repo root or /notebooks ---
def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "raw").is_dir():
            return p
    raise FileNotFoundError("Could not find data/raw above " + str(start))

ROOT = find_root(Path.cwd())
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

print("Project root :", ROOT)
print("Raw folder   :", RAW)
print("Processed    :", PROCESSED)
print("\nFiles in data/raw/:")
for f in sorted(RAW.iterdir()):
    if f.is_file() and not f.name.startswith("."):
        print(f"  {f.stat().st_size/1e6:8.2f} MB  {f.name}")


Project root : /Users/samir/Work Related/JDA-Scrum/Data-Analyst-Mental-Health-Project-
Raw folder   : /Users/samir/Work Related/JDA-Scrum/Data-Analyst-Mental-Health-Project-/data/raw
Processed    : /Users/samir/Work Related/JDA-Scrum/Data-Analyst-Mental-Health-Project-/data/processed

Files in data/raw/:
      6.55 MB  Catalogue Entry Mental health characteristics Ability to handle stress and sources of stress.csv
      0.01 MB  Catalogue Entry Mental health characteristics Ability to handle stress and sources of stress_MetaData.csv
      1.89 MB  Catalogue Entry Mental health characteristics and suicidal thoughts.csv
      0.01 MB  Catalogue Entry Mental health characteristics and suicidal thoughts_MetaData.csv
      1.33 MB  Catalogue Entry Mental health indicators.csv
      0.01 MB  Catalogue Entry Mental health indicators_MetaData.csv
     36.25 MB  Catalogue Entry Perceived health, by gender and province.csv
      0.03 MB  Catalogue Entry Perceived health, by gender and province_M

## 2 · Dataset registry

The 5 datasets the team agreed to use. `kind` tells the profiler how to read each one:

- **statcan_long** – tidy StatCan table (`REF_DATE, GEO, Indicators, ... VALUE`), UTF-8-BOM
- **cihi_vizconfig** – CIHI Health Infobase export; each row is a *chart config*, not a clean record
- **excel_multitable** – CIHI workbook; many formatted tables per sheet, needs custom parsing later
- **microdata** – MHACS survey; one row per respondent, ~600 coded columns


In [2]:
DATASETS = {
    "perceived_mh_annual": {
        "file": "StatCan 13-10-0972 – perceived mental health.csv",
        "owner": "Misa", "kind": "statcan_long",
        "source": "StatCan CCHS – health characteristics, two-year period estimates",
        "note": "Cleanest table. 3 cycles (2019/20, 21/22, 23/24), 13 prov/terr, 8 indicators "
                "(perceived mental health, life stress, mood/anxiety disorder, cannabis, heavy "
                "drinking, belonging). Age = 18+ total only. Best candidate for the dashboard trend view.",
    },
    "suicidal_thoughts": {
        "file": "Catalogue Entry Mental health characteristics and suicidal thoughts.csv",
        "owner": "Fatima", "kind": "statcan_long",
        "source": "StatCan 13-10-0465 subset – mental health characteristics",
        "note": "Only 2 years (2015, 2019) – no post-COVID data. Indicators: suicidal thoughts (15+), "
                "consultation with a professional, positive mental health (flourishing). By age & sex.",
    },
    "stress_coping": {
        "file": "Catalogue Entry Mental health characteristics Ability to handle stress and sources of stress.csv",
        "owner": "Fatima", "kind": "statcan_long",
        "source": "StatCan 13-10-0802 – ability to handle stress / sources of stress",
        "note": "Only 2 years (2016, 2019). ~19% of VALUE rows are blank (suppressed small cells). "
                "Sources of stress: work, finances, family, school, time, health.",
    },
    "perceived_health_quarterly": {
        "file": "Catalogue Entry Mental health indicators.csv",
        "owner": "Fatima", "kind": "statcan_long",
        "source": "StatCan 45-10-0081 – Perceived health, by gender and province",
        "note": "FILENAME IS MISLEADING: this is perceived *general* health (excellent / good / "
                "fair-poor), not mental health. Quarterly 2021-04 → 2023-07. Useful as a control / "
                "context series, not a core mental-health measure.",
    },
    "cchs_mh_disorders": {
        "file": "Catalogue Entry Perceived health, by gender and province.csv",
        "owner": "Fatima", "kind": "statcan_long",
        "source": "StatCan 13-10-0465 – CCHS Mental Health, diagnosed disorders",
        "note": "Richest indicator list (45: anxiety, mood, bipolar, substance use, suicidal thoughts "
                "life/12-mo...) BUT only 3 far-apart cycles (2002, 2012, 2022) and ~39% of VALUE rows "
                "blank. Good for a 2012-vs-2022 comparison, not for a trend line.",
    },
    "cihi_mh_services": {
        "file": "health services for mental illness and alcoholdrug induced disorders.csv",
        "owner": "Fatima", "kind": "cihi_vizconfig",
        "source": "CIHI / PHAC Health Infobase – mental illness & substance-use services",
        "note": "264 rows, but each row is a chart definition (indicator, x_axis_values, y_axis_values). "
                "Needs to be unpivoted into tidy (indicator, breakdown, group, value, ci_low, ci_high) "
                "in step 02 before it is usable.",
    },
    "cihi_children_youth": {
        "file": "care-children-youth-with-mental-disorders-data-tables-en.xlsx",
        "owner": "Fatima", "kind": "excel_multitable",
        "source": "CIHI – Care for Children and Youth With Mental Disorders",
        "note": "32 sheets, most are formatted report tables with title rows & merged cells. The clean "
                "machine-readable data is on the hidden sheets 'Table8DATA_to hide' and "
                "'Table13DATA_to hide' (ED visits / hospitalisations by diagnosis, fiscal 2018/19-2023/24).",
    },
    "mhacs_2022_pumf": {
        "file": "MHACS 2022 Public Use Microdata.csv",
        "owner": "Samir", "kind": "microdata",
        "source": "StatCan – Mental Health and Access to Care Survey 2022, Public Use Microdata File",
        "note": "9,861 respondents x 602 columns, all numeric codes. No blank cells – missing is coded "
                "(6/7/8/9, 96, 996, 99.6 ...). Needs the PUMF data dictionary PDF to decode. Carries "
                "survey weight WTS_M. This is the only dataset that supports regression / ML on "
                "individual-level factors. Key blocks: SUI_* (suicidality), DEP_*/MIA_*/GAD_* "
                "(depression/anxiety), AUD_*/SUD_* (substance use), SPS_* (social support), INCDV* (income).",
    },
}

meta_files = sorted(p.name for p in RAW.glob("*_MetaData.csv"))
print(f"{len(DATASETS)} datasets registered.")
print(f"{len(meta_files)} StatCan metadata files also present:", meta_files)


8 datasets registered.
4 StatCan metadata files also present: ['Catalogue Entry Mental health characteristics Ability to handle stress and sources of stress_MetaData.csv', 'Catalogue Entry Mental health characteristics and suicidal thoughts_MetaData.csv', 'Catalogue Entry Mental health indicators_MetaData.csv', 'Catalogue Entry Perceived health, by gender and province_MetaData.csv']


## 3 · Loaders

In [3]:
def load_dataset(key: str):
    """Return a DataFrame for CSV-based datasets. Excel is profiled separately (see section 5)."""
    spec = DATASETS[key]
    path = RAW / spec["file"]
    if not path.exists():
        raise FileNotFoundError(path)

    if spec["kind"] in ("statcan_long", "cihi_vizconfig"):
        # StatCan CSVs carry a UTF-8 BOM; low_memory=False avoids mixed-type warnings
        return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    if spec["kind"] == "microdata":
        return pd.read_csv(path, low_memory=False)
    raise ValueError(f"{key}: use section 5 for kind={spec['kind']}")


# quick smoke test
_df = load_dataset("perceived_mh_annual")
print(_df.shape)
_df.head(3)


(936, 18)


,REF_DATE,GEO,DGUID,Age group,Sex,Indicators,Characteristics,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2019/2020,Newfoundland and Labrador,2021A000210,"Total, 18 years and over",Both sexes,"Perceived mental health, very good or excellent",Percent,Percent,239,units,0,v1806947204,2.1.1.3.4,68.6,NaN,NaN,NaN,1
1,2021/2022,Newfoundland and Labrador,2021A000210,"Total, 18 years and over",Both sexes,"Perceived mental health, very good or excellent",Percent,Percent,239,units,0,v1806947204,2.1.1.3.4,58.3,NaN,NaN,NaN,1
2,2023/2024,Newfoundland and Labrador,2021A000210,"Total, 18 years and over",Both sexes,"Perceived mental health, very good or excellent",Percent,Percent,239,units,0,v1806947204,2.1.1.3.4,53.5,NaN,NaN,NaN,1


## 4 · Profile every CSV dataset

For each dataset we record: shape, memory, duplicate rows, overall % missing, and –
for StatCan long tables – the time span, the geographies, and the indicator list.
Per-column detail goes into `column_profiles`.

In [4]:
STATCAN_DIMS = ["REF_DATE", "GEO", "Age group", "Sex", "Gender",
                "Indicators", "Characteristics", "Statistics", "UOM"]

def ref_date_span(s: pd.Series):
    vals = sorted(s.dropna().astype(str).unique())
    return (vals[0], vals[-1], len(vals)) if vals else (None, None, 0)

def profile_csv(key: str):
    spec = DATASETS[key]
    df = load_dataset(key)

    n_missing_cells = int(df.isna().sum().sum())
    pct_missing = round(100 * n_missing_cells / (df.shape[0] * df.shape[1]), 2)

    row = {
        "dataset": key,
        "owner": spec["owner"],
        "kind": spec["kind"],
        "source": spec["source"],
        "file": spec["file"],
        "rows": df.shape[0],
        "cols": df.shape[1],
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1e6, 2),
        "duplicate_rows": int(df.duplicated().sum()),
        "pct_cells_missing": pct_missing,
        "pct_value_missing": (round(100 * df["VALUE"].isna().mean(), 1)
                              if "VALUE" in df.columns else None),
        "ref_date_min": None, "ref_date_max": None, "n_periods": None,
        "n_geo": None, "n_indicators": None,
        "notes": spec["note"],
    }
    if "REF_DATE" in df.columns:
        row["ref_date_min"], row["ref_date_max"], row["n_periods"] = ref_date_span(df["REF_DATE"])
    if "GEO" in df.columns:
        row["n_geo"] = df["GEO"].nunique()
    if "Indicators" in df.columns:
        row["n_indicators"] = df["Indicators"].nunique()
    elif "indicator" in df.columns:
        row["n_indicators"] = df["indicator"].nunique()

    # per-column profile
    col_rows = []
    for col in df.columns:
        s = df[col]
        uniq = s.dropna().unique()
        sample = ", ".join(map(lambda v: str(v)[:30], uniq[:4]))
        col_rows.append({
            "dataset": key,
            "column": col,
            "dtype": str(s.dtype),
            "n_missing": int(s.isna().sum()),
            "pct_missing": round(100 * s.isna().mean(), 1),
            "n_unique": int(s.nunique(dropna=True)),
            "sample_values": sample,
        })
    return row, pd.DataFrame(col_rows), df


inventory_rows = []
column_profiles = []
frames = {}

for key, spec in DATASETS.items():
    if spec["kind"] == "excel_multitable":
        continue
    print(f"\n{'='*90}\n{key}   ({spec['owner']})\n{'='*90}")
    row, cols_df, df = profile_csv(key)
    inventory_rows.append(row)
    column_profiles.append(cols_df)
    frames[key] = df

    print(f"shape          : {row['rows']:,} rows x {row['cols']} cols   ({row['memory_mb']} MB in memory)")
    print(f"time coverage  : {row['ref_date_min']} -> {row['ref_date_max']}  ({row['n_periods']} periods)")
    print(f"geographies    : {row['n_geo']}")
    print(f"indicators     : {row['n_indicators']}")
    print(f"duplicate rows : {row['duplicate_rows']}")
    print(f"cells missing  : {row['pct_cells_missing']}%")
    dims = [d for d in STATCAN_DIMS if d in df.columns]
    for d in dims:
        vals = sorted(map(str, df[d].dropna().unique()))
        shown = vals if len(vals) <= 12 else vals[:12] + [f"... (+{len(vals)-12} more)"]
        print(f"  {d:15}: {shown}")



perceived_mh_annual   (Misa)
shape          : 936 rows x 18 cols   (0.71 MB in memory)
time coverage  : 2019/2020 -> 2023/2024  (3 periods)
geographies    : 13
indicators     : 8
duplicate rows : 0
cells missing  : 16.6%
  REF_DATE       : ['2019/2020', '2021/2022', '2023/2024']
  GEO            : ['Alberta', 'British Columbia', 'Manitoba', 'New Brunswick', 'Newfoundland and Labrador', 'Northwest Territories', 'Nova Scotia', 'Nunavut', 'Ontario', 'Prince Edward Island', 'Quebec', 'Saskatchewan', '... (+1 more)']
  Age group      : ['Total, 18 years and over']
  Sex            : ['Both sexes', 'Females', 'Males']
  Indicators     : ['Anxiety disorder', 'Cannabis use, past 12 months', 'Heavy drinking', 'Mood disorder', 'Perceived life stress, most days quite a bit or extremely stressful', 'Perceived mental health, fair or poor', 'Perceived mental health, very good or excellent', 'Sense of belonging to local community, somewhat strong or very strong']
  Characteristics: ['Percent']
  UOM

shape          : 8,208 rows x 18 cols   (6.12 MB in memory)
time coverage  : 2015 -> 2019  (2 periods)
geographies    : 11
indicators     : 3
duplicate rows : 0
cells missing  : 15.57%
  REF_DATE       : ['2015', '2019']
  GEO            : ['Alberta', 'British Columbia', 'Canada (excluding territories)', 'Manitoba', 'New Brunswick', 'Newfoundland and Labrador', 'Nova Scotia', 'Ontario', 'Prince Edward Island', 'Quebec', 'Saskatchewan']
  Age group      : ['12 to 17 years', '18 to 34 years', '35 to 49 years', '50 to 64 years', '65 years and over', 'Total, 12 years and over']
  Sex            : ['Both sexes', 'Females', 'Males']
  Indicators     : ['Consultation with a health professional about emotional or mental health', 'Positive mental health, flourishing', 'Suicidal thoughts (15 years and over)']
  Characteristics: ['High 95% confidence interval, number of persons', 'High 95% confidence interval, percent', 'Low 95% confidence interval, number of persons', 'Low 95% confidence interva

shape          : 27,360 rows x 18 cols   (20.72 MB in memory)
time coverage  : 2016 -> 2019  (2 periods)
geographies    : 11
indicators     : 10
duplicate rows : 0
cells missing  : 15.75%
  REF_DATE       : ['2016', '2019']
  GEO            : ['Alberta', 'British Columbia', 'Canada (excluding territories)', 'Manitoba', 'New Brunswick', 'Newfoundland and Labrador', 'Nova Scotia', 'Ontario', 'Prince Edward Island', 'Quebec', 'Saskatchewan']
  Age group      : ['12 to 17 years', '18 to 34 years', '35 to 49 years', '50 to 64 years', '65 years and over', 'Total, 12 years and over']
  Sex            : ['Both sexes', 'Females', 'Males']
  Indicators     : ['Ability to handle the day-to-day demands in life, good or excellent', 'Ability to handle unexpected and difficult problems, good or excellent', 'Main source of stress in day-to-day life, family', 'Main source of stress in day-to-day life, financial concerns', 'Main source of stress in day-to-day life, health', 'Main source of stress in day

shape          : 160,992 rows x 18 cols   (120.3 MB in memory)
time coverage  : 2002 -> 2022  (3 periods)
geographies    : 13
indicators     : 45
duplicate rows : 0
cells missing  : 15.68%
  REF_DATE       : ['2002', '2012', '2022']
  GEO            : ['Alberta', 'Atlantic Provinces', 'British Columbia', 'Canada', 'Manitoba', 'New Brunswick', 'Newfoundland and Labrador', 'Nova Scotia', 'Ontario', 'Prairie Provinces', 'Prince Edward Island', 'Quebec', '... (+1 more)']
  Age group      : ['15 to 24 years', '25 to 44 years', '25 to 64 years', '45 to 64 years', '65 years and over', 'Total, 15 years and over']
  Gender         : ['Men+', 'Total, gender of person', 'Women+']
  Indicators     : ['Alcohol abuse or dependence, 12 months', 'Alcohol abuse or dependence, life', 'Any anxiety disorder, 12 months', 'Any anxiety disorder, life', 'Any mood disorder, 12 months', 'Any mood disorder, life', 'Any selected disorder (mental or substance), excluding social phobia, 12 months', 'Any selected di

shape          : 9,861 rows x 602 cols   (47.49 MB in memory)
time coverage  : None -> None  (None periods)
geographies    : None
indicators     : None
duplicate rows : 0
cells missing  : 0.0%


## 5 · Profile the CIHI children/youth Excel workbook

This workbook is not analysis-ready. We only enumerate its sheets here and point step 02
at the two hidden data sheets.

In [5]:
xl_spec = DATASETS["cihi_children_youth"]
xl_path = RAW / xl_spec["file"]
xls = pd.ExcelFile(xl_path)

data_sheets = [s for s in xls.sheet_names if s.endswith("_to hide")]
print(f"{len(xls.sheet_names)} sheets total")
print("machine-readable data sheets:", data_sheets)

xl_preview = {}
for s in data_sheets:
    raw = xls.parse(s, header=None)
    # row 0 is a title, row 1 holds the real headers
    hdr = raw.iloc[1].tolist()
    body = raw.iloc[2:].reset_index(drop=True)
    body.columns = [str(h).replace("\n", " ").strip() for h in hdr]
    xl_preview[s] = body
    print(f"\n--- {s}: {body.shape[0]} rows x {body.shape[1]} cols")
    print("columns:", list(body.columns))
    display(body.head(4))

inventory_rows.append({
    "dataset": "cihi_children_youth", "owner": xl_spec["owner"], "kind": xl_spec["kind"],
    "source": xl_spec["source"], "file": xl_spec["file"],
    "rows": sum(v.shape[0] for v in xl_preview.values()),
    "cols": max(v.shape[1] for v in xl_preview.values()),
    "memory_mb": round(xl_path.stat().st_size / 1e6, 2),
    "duplicate_rows": None, "pct_cells_missing": None, "pct_value_missing": None,
    "ref_date_min": "2018-2019", "ref_date_max": "2023-2024", "n_periods": 6,
    "n_geo": None, "n_indicators": None, "notes": xl_spec["note"],
})


32 sheets total
machine-readable data sheets: ['Table8DATA_to hide', 'Table13DATA_to hide']

--- Table8DATA_to hide: 216 rows x 13 cols
columns: ['Fiscal year', 'Diagnosis category', 'Sex', 'Age group: 5–9 years – Rate', 'Age group: 5–9 years – 95% CI', 'Age group: 10–14 years – Rate', 'Age group: 10–14 years – 95% CI', 'Age group: 15–17 years – Rate', 'Age group: 15–17 years – 95% CI', 'Age group: 18–24 years – Rate', 'Age group: 18–24 years – 95% CI', 'Age group: 5–24 years – Rate', 'Age group: 5–24 years – 95% CI']


,Fiscal year,Diagnosis category,Sex,Age group: 5–9 years – Rate,Age group: 5–9 years – 95% CI,Age group: 10–14 years – Rate,Age group: 10–14 years – 95% CI,Age group: 15–17 years – Rate,Age group: 15–17 years – 95% CI,Age group: 18–24 years – Rate,Age group: 18–24 years – 95% CI,Age group: 5–24 years – Rate,Age group: 5–24 years – 95% CI
0,2018–2019,Neurocognitive disorders,Female,6,4–8,21,17–25,52,44–60,35,31–39,27,25–29
1,2018–2019,Neurocognitive disorders,Male,9,7–12,17,13–20,31,25–37,23,20–27,20,18–21
2,2018–2019,Neurocognitive disorders,Total,8,6–9,19,16–21,41,36–46,29,26–31,23,22–25
3,2018–2019,Substance-related disorders,Female,1,0–2,74,66–81,513,488–538,738,720–756,376,368–385



--- Table13DATA_to hide: 216 rows x 13 cols
columns: ['Fiscal year', 'Diagnosis category', 'Sex', 'Age group: 5–9 years – Rate', 'Age group: 5–9 years – 95% CI', 'Age group: 10–14 years – Rate', 'Age group: 10–14 years – 95% CI', 'Age group: 15–17 years – Rate', 'Age group: 15–17 years – 95% CI', 'Age group: 18–24 years – Rate', 'Age group: 18–24 years – 95% CI', 'Age group: 5–24 years – Rate', 'Age group: 5–24 years – 95% CI']


,Fiscal year,Diagnosis category,Sex,Age group: 5–9 years – Rate,Age group: 5–9 years – 95% CI,Age group: 10–14 years – Rate,Age group: 10–14 years – 95% CI,Age group: 15–17 years – Rate,Age group: 15–17 years – 95% CI,Age group: 18–24 years – Rate,Age group: 18–24 years – 95% CI,Age group: 5–24 years – Rate,Age group: 5–24 years – 95% CI
0,2018–2019,Neurocognitive disorders,Female,0.3,0–1,1,0–2,2.2,1–3,2.2,1–3,1.5,1–2
1,2018–2019,Neurocognitive disorders,Male,0.3,0–1,0.7,0–1,1.5,1–2,2.7,2–3,1.5,1–2
2,2018–2019,Neurocognitive disorders,Total,0.3,0–1,0.8,0–1,1.8,1–3,2.4,2–3,1.5,1–2
3,2018–2019,Substance-related disorders,Female,0.1,0–0,10.6,9–13,79.3,72–87,109.1,104–114,55.8,53–58


## 6 · Build the inventory and write outputs to `data/processed/`

In [6]:
COL_ORDER = ["dataset", "owner", "kind", "rows", "cols", "n_periods",
             "ref_date_min", "ref_date_max", "n_geo", "n_indicators",
             "pct_value_missing", "pct_cells_missing", "duplicate_rows",
             "memory_mb", "source", "file", "notes"]

inventory = pd.DataFrame(inventory_rows)[COL_ORDER].sort_values("dataset").reset_index(drop=True)
profiles = pd.concat(column_profiles, ignore_index=True)

inv_path = PROCESSED / "01_data_inventory.csv"
prof_path = PROCESSED / "01_column_profiles.csv"
inventory.to_csv(inv_path, index=False)
profiles.to_csv(prof_path, index=False)

print("wrote", inv_path.relative_to(ROOT), f"({inventory.shape[0]} rows)")
print("wrote", prof_path.relative_to(ROOT), f"({profiles.shape[0]} rows)")
inventory


wrote data/processed/01_data_inventory.csv (8 rows)
wrote data/processed/01_column_profiles.csv (705 rows)


,dataset,owner,kind,rows,cols,n_periods,ref_date_min,ref_date_max,n_geo,n_indicators,pct_value_missing,pct_cells_missing,duplicate_rows,memory_mb,source,file,notes
0,cchs_mh_disorders,Fatima,statcan_long,160992,18,3.0,2002,2022,13.0,45.0,38.5,15.68,0.0,120.30,"StatCan 13-10-0465 – CCHS Mental Health, diagn...","Catalogue Entry Perceived health, by gender an...","Richest indicator list (45: anxiety, mood, bip..."
1,cihi_children_youth,Fatima,excel_multitable,432,13,6.0,2018-2019,2023-2024,NaN,NaN,NaN,NaN,NaN,1.19,CIHI – Care for Children and Youth With Mental...,care-children-youth-with-mental-disorders-data...,"32 sheets, most are formatted report tables wi..."
2,cihi_mh_services,Fatima,cihi_vizconfig,264,14,NaN,NaN,NaN,NaN,29.0,NaN,28.11,0.0,0.21,CIHI / PHAC Health Infobase – mental illness &...,health services for mental illness and alcohol...,"264 rows, but each row is a chart definition (..."
3,mhacs_2022_pumf,Samir,microdata,9861,602,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.0,47.49,StatCan – Mental Health and Access to Care Sur...,MHACS 2022 Public Use Microdata.csv,"9,861 respondents x 602 columns, all numeric c..."
4,perceived_health_quarterly,Fatima,statcan_long,6318,17,9.0,2021-04,2023-07,13.0,3.0,5.5,16.94,0.0,4.57,"StatCan 45-10-0081 – Perceived health, by gend...",Catalogue Entry Mental health indicators.csv,FILENAME IS MISLEADING: this is perceived *gen...
5,perceived_mh_annual,Misa,statcan_long,936,18,3.0,2019/2020,2023/2024,13.0,8.0,2.8,16.60,0.0,0.71,"StatCan CCHS – health characteristics, two-yea...",StatCan 13-10-0972 – perceived mental health.csv,"Cleanest table. 3 cycles (2019/20, 21/22, 23/2..."
6,stress_coping,Fatima,statcan_long,27360,18,2.0,2016,2019,11.0,10.0,18.9,15.75,0.0,20.72,StatCan 13-10-0802 – ability to handle stress ...,Catalogue Entry Mental health characteristics ...,"Only 2 years (2016, 2019). ~19% of VALUE rows ..."
7,suicidal_thoughts,Fatima,statcan_long,8208,18,2.0,2015,2019,11.0,3.0,6.9,15.57,0.0,6.12,StatCan 13-10-0465 subset – mental health char...,Catalogue Entry Mental health characteristics ...,"Only 2 years (2015, 2019) – no post-COVID data..."


In [7]:
# pipeline marker – proves data/raw -> data/processed works end to end
check_path = PROCESSED / "01_pipeline_check.txt"
lines = [
    "01_data_understanding.ipynb - pipeline check",
    f"run_at        : {datetime.now().isoformat(timespec='seconds')}",
    f"project_root  : {ROOT}",
    f"datasets_read : {len(frames) + 1}",
    f"csv_datasets  : {len(frames)}",
    f"excel_sheets  : {len(data_sheets)}",
    f"total_rows    : {int(inventory['rows'].fillna(0).sum()):,}",
    f"outputs       : {inv_path.name}, {prof_path.name}, {check_path.name}",
]
check_path.write_text("\n".join(lines) + "\n")
print("\n".join(lines))


01_data_understanding.ipynb - pipeline check
run_at        : 2026-08-26T23:54:12
project_root  : /Users/samir/Work Related/JDA-Scrum/Data-Analyst-Mental-Health-Project-
datasets_read : 8
csv_datasets  : 7
excel_sheets  : 2
total_rows    : 214,371
outputs       : 01_data_inventory.csv, 01_column_profiles.csv, 01_pipeline_check.txt


In [8]:
# --- assertions: the "test" ---
assert inv_path.exists() and inv_path.stat().st_size > 0, "inventory not written"
assert prof_path.exists() and prof_path.stat().st_size > 0, "column profiles not written"
assert check_path.exists(), "pipeline check not written"
assert inventory["dataset"].is_unique, "duplicate dataset keys"
assert len(inventory) == len(DATASETS), "some datasets missing from inventory"
assert frames["mhacs_2022_pumf"].shape == (9861, 602), "MHACS shape changed unexpectedly"

print("PIPELINE TEST PASSED - data/raw -> data/processed is working")
print(f"   {len(inventory)} datasets profiled, {len(profiles)} columns catalogued")


PIPELINE TEST PASSED - data/raw -> data/processed is working
   8 datasets profiled, 705 columns catalogued


## 7 · Findings & decisions for the team

### Time coverage is the main constraint
| Dataset | Periods available | Trend line possible? |
|---|---|---|
| `perceived_mh_annual` (Misa) | 2019/20, 2021/22, 2023/24 | Weak (3 points) – **best we have** |
| `perceived_health_quarterly` | 2021-Q2 → 2023-Q3 (10 qtrs) | Yes, but it's *general* health |
| `suicidal_thoughts` | 2015, 2019 | No |
| `stress_coping` | 2016, 2019 | No |
| `cchs_mh_disorders` | 2002, 2012, 2022 | No – comparison only |
| `cihi_children_youth` | 2018/19 → 2023/24 (6 fiscal yrs) | Yes (children/youth only) |

➡️ **A single Canada-wide "mental-health over time" line chart is not well supported.**
Either (a) reframe around *cycle-to-cycle comparison* + *provincial differences*, or (b) the
team pulls one long annual StatCan series via the WDS API (e.g. self-rated mental health, or
suicide deaths 13-10-0392) to anchor the trend.

### Data quality flags
- `perceived_health_quarterly` filename says "mental health indicators" but the data is **general perceived health**. Rename / relabel.
- `cchs_mh_disorders` and `stress_coping` have **large blocks of suppressed values** (39% / 19%) – small-cell confidentiality. Keep the CI columns; don't impute.
- `cihi_mh_services` is **chart-config, not tidy data** – must be unpivoted in step 02.
- `cihi_children_youth` – use only the two `*_to hide` sheets; ignore the 30 formatted sheets.
- `mhacs_2022_pumf` – **do not touch before decoding** with the PUMF data-dictionary PDF; every value is a code and missing is coded, not blank.

### Two analysis tracks (they do not join)
- **Track A – aggregated (all StatCan + CIHI):** province / cycle / age / sex → KPIs, maps, comparisons, dashboard.
- **Track B – MHACS microdata:** individual-level → "which factors are associated with poor mental health / suicidal ideation" → regression, feature importance, optional ML.
- Shared key for Track A: **GEO + REF_DATE (+ Age group + Sex)**.

### Next steps
1. Team signs off on the reframed question (comparison + provincial, not national trend) — or agrees to pull one long series via the WDS API.
2. `02_data_cleaning.ipynb`: melt the StatCan tables to tidy long, standardise GEO names, split Number vs Percent vs CI into columns, filter to the agreed window, write to `data/processed/`.
3. Decode ~25 MHACS variables into `docs/data_dictionary.md` (Samir).
4. Parse the two CIHI Excel data sheets into tidy CSVs.
